# Cattle Re-ID — Etapa 3: comparación de losses (CE / ArcFace / SupCon / Triplet)

Misma augmentation fuerte + backbone + sampler PK en las 4; se entrena en **CMPD300**, se **congela**, y se evalúa clustering en **Zenodo** (sin etiquetas). Métrica primaria: **HDBSCAN ARI**, **una corrida por loss** (sin sweep de seeds). Baselines: ImageNet y DINOv2.

**Modelos cacheados a Drive** (celda 2): DINOv2 y ResNet-ImageNet se descargan una sola vez. Los checkpoints entrenados se guardan en `outputs/checkpoints` (symlink a Drive) → persisten.

## 1. Montar Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/cattle_reid')   # ajustá si tu carpeta se llama distinto
assert DRIVE_ROOT.is_dir(), f'No existe {DRIVE_ROOT}.'
print('contenido:', [p.name for p in DRIVE_ROOT.iterdir()])

Mounted at /content/drive
contenido: ['BeefCattle_Muzzle_Individualized.zip', 'outputs', 'dataset_caras_bovinos.zip', 'gradcam_test', 'CMPD300_Baseline.zip', 'Untitled0.ipynb']


## 2. Cachear modelos en Drive (ANTES de importar torch)

Setea `TORCH_HOME` (ResNet ImageNet) y `HF_HOME` (DINOv2) apuntando a Drive: se descargan una vez y se reusan en cada sesión.

In [2]:
import os
CACHE = DRIVE_ROOT / 'model_cache'
(CACHE/'torch').mkdir(parents=True, exist_ok=True)
(CACHE/'hf').mkdir(parents=True, exist_ok=True)
os.environ['TORCH_HOME'] = str(CACHE/'torch')
os.environ['HF_HOME']    = str(CACHE/'hf')
print('TORCH_HOME=', os.environ['TORCH_HOME'])
print('HF_HOME   =', os.environ['HF_HOME'])

TORCH_HOME= /content/drive/MyDrive/cattle_reid/model_cache/torch
HF_HOME   = /content/drive/MyDrive/cattle_reid/model_cache/hf


## 3. GPU

In [3]:
!nvidia-smi -L
import torch
assert torch.cuda.is_available(), 'Sin GPU: Entorno -> Cambiar tipo -> GPU.'
print('GPU:', torch.cuda.get_device_name(0))

GPU 0: Tesla T4 (UUID: GPU-62cd21e1-9954-5336-0b3a-5d9602a52baf)
GPU: Tesla T4


## 4. Código + dependencias

In [4]:
import shutil
os.chdir('/content')
REPO_URL = 'https://github.com/benjaminvitale/tp-final-vision2-Pujol-Vitale.git'
BRANCH   = 'main'
REPO_DIR = '/content/tp-final-vision2-Pujol-Vitale'
if os.path.exists(REPO_DIR): shutil.rmtree(REPO_DIR)
!git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -1
!pip -q install 'transformers==4.40.2'

Cloning into '/content/tp-final-vision2-Pujol-Vitale'...
remote: Enumerating objects: 337, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 337 (delta 9), reused 13 (delta 6), pack-reused 305 (from 1)
Receiving objects: 100% (337/337), 296.56 KiB | 1.59 MiB/s, done.
Resolving deltas: 100% (184/184), done.
/content/tp-final-vision2-Pujol-Vitale
45921e7 (HEAD -> main, origin/main, origin/HEAD) Etapa 3: notebook de losses robusto para corrida nocturna desatendida
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 73.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following 

## 5. Persistir outputs en Drive (checkpoints entrenados no se pierden)

In [5]:
DRIVE_OUTPUTS = DRIVE_ROOT / 'outputs'
for sub in ('checkpoints', 'logs', 'results'):
    ds = DRIVE_OUTPUTS / sub; ds.mkdir(parents=True, exist_ok=True)
    ls = Path(REPO_DIR) / 'outputs' / sub
    if ls.exists() and not ls.is_symlink(): shutil.rmtree(ls)
    if not ls.exists(): os.symlink(ds, ls, target_is_directory=True)
    print(f'outputs/{sub} -> {ds}')

outputs/checkpoints -> /content/drive/MyDrive/cattle_reid/outputs/checkpoints
outputs/logs -> /content/drive/MyDrive/cattle_reid/outputs/logs
outputs/results -> /content/drive/MyDrive/cattle_reid/outputs/results


## 6. Datos: CMPD300 (source, entrenamiento) + Zenodo (target, eval)

In [7]:
import zipfile
IMG_EXTS = {'.jpg','.jpeg','.png','.bmp','.webp'}

# --- CMPD300 (source) ---
CMPD_ZIP = DRIVE_ROOT / 'CMPD300_Baseline.zip'
CMPD_LOCAL = Path('/content/cmpd300')
if not CMPD_LOCAL.exists():
    CMPD_LOCAL.mkdir(parents=True)
    !unzip -q "{CMPD_ZIP}" -d "{CMPD_LOCAL}"
cand = [CMPD_LOCAL, CMPD_LOCAL/'Baseline'] + [p for p in CMPD_LOCAL.rglob('*') if p.is_dir()]
CMPD_DIR = next((p for p in cand if (p/'train').is_dir()), None)
assert CMPD_DIR, 'No encuentro train/ en CMPD300.'
SOURCE_TRAIN = CMPD_DIR / 'train'
print('SOURCE_TRAIN:', SOURCE_TRAIN, '| ids:', len([d for d in SOURCE_TRAIN.iterdir() if d.is_dir()]))

# --- Zenodo (target) ---
MUZZLE_ZIP = DRIVE_ROOT / 'BeefCattle_Muzzle_Individualized.zip'
MUZZLE_LOCAL = Path('/content/muzzle')
if not MUZZLE_LOCAL.exists():
    MUZZLE_LOCAL.mkdir(parents=True)
    !unzip -q "{MUZZLE_ZIP}" -d "{MUZZLE_LOCAL}"
def find_root(base):
    for p in [base, *[d for d in base.rglob('*') if d.is_dir()]]:
        subs = [d for d in p.iterdir() if d.is_dir()]
        if len(subs) >= 100 and any(d.name.lower().startswith('cattle') for d in subs): return p
    raise RuntimeError('No encuentro cattle_*')
TARGET_DIR = find_root(MUZZLE_LOCAL)
print('TARGET_DIR:', TARGET_DIR, '| ids:', len([d for d in TARGET_DIR.iterdir() if d.is_dir()]))

SOURCE_TRAIN: /content/cmpd300/MuzzleSplit/train | ids: 300
TARGET_DIR: /content/muzzle/BeefCattle_Muzzle_Individualized | ids: 268


## 7. Validación rápida del pipeline (smoke, ~1 min)

Antes de quemar GPU: confirma que las 4 losses entrenan y guardan checkpoint.

In [8]:
for L in ['ce','arcface','supcon','triplet']:
    print('==== smoke', L, '====')
    !python scripts/13_train_loss.py --train-dir "{SOURCE_TRAIN}" --loss {L} --smoke --out /tmp/smoke_{L}.pt 2>&1 | tail -2

==== smoke ce ====
[01:56:51] INFO ep 02/2 | loss 5.4341 | train acc 0.0104 | lr 1.38e-04
[01:56:52] INFO saved ce encoder → /tmp/smoke_ce.pt | {'loss': 'ce', 'seed': 0, 'num_classes': 300, 'epochs': 2, 'P': 16, 'K': 4, 'image_size': 224, 'checkpoint': '/tmp/smoke_ce.pt', 'elapsed_sec': 15.9}
==== smoke arcface ====
[01:57:17] INFO ep 02/2 | loss 19.9900 | train acc 0.0000 | lr 1.38e-04
[01:57:18] INFO saved arcface encoder → /tmp/smoke_arcface.pt | {'loss': 'arcface', 'seed': 0, 'num_classes': 300, 'epochs': 2, 'P': 16, 'K': 4, 'image_size': 224, 'checkpoint': '/tmp/smoke_arcface.pt', 'elapsed_sec': 16.4}
==== smoke supcon ====
[01:57:38] INFO ep 02/2 | loss 4.1492 | lr 1.38e-04
[01:57:39] INFO saved supcon encoder → /tmp/smoke_supcon.pt | {'loss': 'supcon', 'seed': 0, 'num_classes': 300, 'epochs': 2, 'P': 16, 'K': 4, 'image_size': 224, 'checkpoint': '/tmp/smoke_supcon.pt', 'elapsed_sec': 16.1}
==== smoke triplet ====
[01:58:01] INFO ep 02/2 | loss 0.7196 | lr 1.38e-04
[01:58:02] INFO

## 8. Entrenar las 4 losses

Cada loss = un entrenamiento full de ResNet-50 sobre CMPD300 (~1–2 h en T4). Los checkpoints ya hechos se saltean, así que podés reanudar si se corta.

In [10]:
# Entrena las 4 losses. RESILIENTE para correr de noche sin supervisión:
#  - saltea las que ya tienen checkpoint en Drive (reanudable si se corta la sesión);
#  - si una loss FALLA, la registra y SIGUE con las demás (no se cae todo el loop);
#  - el log de cada entrenamiento se escribe a Drive (outputs/logs/train_<loss>.log)
#    ademas de mostrarse en vivo, asi a la mañana podes ver que paso aunque el tab se haya caido.
import subprocess
from pathlib import Path

LOSSES = ['ce', 'arcface', 'supcon', 'triplet']
EPOCHS = 60
LOGDIR = Path('outputs/logs'); LOGDIR.mkdir(parents=True, exist_ok=True)

done, failed = [], []
for loss in LOSSES:
    out = Path('outputs/checkpoints') / f'cmpd300_{loss}.pt'
    if out.exists():
        print('✓ ya existe, salteo:', out.name); done.append(loss); continue
    logf = LOGDIR / f'train_{loss}.log'
    print(f'\n==== entrenando {loss}  (log en Drive: {logf}) ====', flush=True)
    with open(logf, 'w') as fh:
        p = subprocess.Popen(
            ['python', '-u', 'scripts/13_train_loss.py', '--train-dir', str(SOURCE_TRAIN),
             '--loss', loss, '--epochs', str(EPOCHS), '--out', str(out)],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:          # tee: notebook en vivo + archivo en Drive
            print(line, end=''); fh.write(line); fh.flush()
        p.wait()
    if p.returncode == 0 and out.exists():
        print(f'   ✓ {loss} guardado → {out}'); done.append(loss)
    else:
        print(f'   ✗ {loss} FALLO (rc={p.returncode}) — revisar {logf}'); failed.append(loss)

print('\n' + '=' * 50)
print('RESUMEN entrenamiento:  ok =', done, ' | fallaron =', failed)
if failed:
    print('Para reintentar los que fallaron: volve a correr ESTA celda (saltea los ya hechos).')


==== entrenando ce  (log en Drive: outputs/logs/train_ce.log) ====
[01:59:07] INFO loss=ce | device=cuda | ids=300 | imgs=1500 | P=16 K=4 | epochs=60 | image_size=224 | norm=True
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader runni

## 9. Evaluar (dedup pHash → HDBSCAN ARI + baselines, una corrida por loss)

In [11]:
!python scripts/14_eval_losses.py \
    --target-dir "{TARGET_DIR}" \
    --ckpt-dir outputs/checkpoints \
    --losses ce arcface supcon triplet \
    --baselines imagenet dinov2

[03:23:05] INFO target=/content/muzzle/BeefCattle_Muzzle_Individualized | 268 ids | 4923 imgs
[03:23:27] INFO pHash dedup: {'n_input': 4923, 'n_kept': 1554, 'n_dropped': 3369, 'threshold': 6}
[03:23:27] INFO eval set: 1554 imgs | gallery=260 probe=1286
[03:23:44] INFO encoder='imagenet_resnet50' embedded 1554 imgs → dim 2048
[03:24:31] INFO baseline imagenet: {'hdbscan_ari': 0.461, 'hdbscan_nmi': 0.9291, 'n_clusters': 373, 'kmeans_ari': 0.7373, 'rank1': 0.8033, 'mAP': 0.8477}
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
config.json: 100% 548/548 [00:00<00:00, 3.23MB/s]
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume

## 10. Resultados

In [12]:
import json
res = json.loads(Path('outputs/results/14_loss_comparison.json').read_text())
print('eval set:', res['n_eval_images'], 'imgs (pHash:', res['phash'], ')')
print('\nbaselines:')
for b,m in res['baselines'].items():
    print(f"  {b:12} HDBSCAN ARI={m.get('hdbscan_ari')}  kmeans={m.get('kmeans_ari')}")
ce = res['per_loss'].get('ce',{}).get('hdbscan_ari')
print('\nlosses (HDBSCAN ARI):')
for loss,m in res['per_loss'].items():
    d = f"  Δce={m['hdbscan_ari']-ce:+.3f}" if ce is not None else ''
    print(f"  {loss:12} {m['hdbscan_ari']}  kmeans={m['kmeans_ari']}  Rank-1={m['rank1']}{d}")

eval set: 1554 imgs (pHash: True )

baselines:
  imagenet     HDBSCAN ARI=0.461  kmeans=0.7373
  dinov2       HDBSCAN ARI=0.1504  kmeans=0.5736

losses (HDBSCAN ARI):
  ce           0.4916  kmeans=0.6935  Rank-1=0.8002  Δce=+0.000
  arcface      0.2674  kmeans=0.6284  Rank-1=0.7208  Δce=-0.224
  supcon       0.5423  kmeans=0.7427  Rank-1=0.8095  Δce=+0.051
  triplet      0.2101  kmeans=0.5879  Rank-1=0.689  Δce=-0.281


## Cómo leer

- **HDBSCAN ARI** es la vara (no Rank-1: el probe de borde mostró que el Rank-1 se infla con contexto/fondo).
- **Δce** aísla el efecto de la loss del efecto de la augmentation. Si CE ya sube fuerte y las demás casi no se diferencian → *lo que importa es forzar el foco en el hocico, no la loss* (hallazgo igual de válido).
- Una loss "gana" solo si supera a ImageNet/DINOv2.

In [13]:
# === VERIFICACIÓN FINAL: ¿quedó TODO guardado en Drive? ===
# Corré esto a la mañana (o al final del run) para confirmar que no se perdió nada.
# Lee directo de Drive, no del symlink, para chequear la persistencia real.
CKPT_DIR = DRIVE_ROOT / 'outputs' / 'checkpoints'
RES_DIR  = DRIVE_ROOT / 'outputs' / 'results'
LOG_DIR  = DRIVE_ROOT / 'outputs' / 'logs'

print('Checkpoints en Drive:')
ok = True
for loss in ['ce', 'arcface', 'supcon', 'triplet']:
    f = CKPT_DIR / f'cmpd300_{loss}.pt'
    if f.exists():
        print(f'   ✓ {f.name}  ({f.stat().st_size / 1e6:.0f} MB)')
    else:
        print(f'   ✗ FALTA {f.name}'); ok = False

rj = RES_DIR / '14_loss_comparison.json'
print('\nResultados en Drive:')
print(f'   {"✓" if rj.exists() else "✗ FALTA"}  {rj.name}')
ok = ok and rj.exists()

print('\nLogs de entrenamiento en Drive:', [p.name for p in sorted(LOG_DIR.glob("train_*.log"))])
print('\n' + ('✅ TODO GUARDADO — podés cerrar Colab tranquilo.'
              if ok else '⚠️  FALTA algo arriba — revisá los logs / reejecutá las celdas 8 y 9.'))

Checkpoints en Drive:
   ✓ cmpd300_ce.pt  (97 MB)
   ✓ cmpd300_arcface.pt  (97 MB)
   ✓ cmpd300_supcon.pt  (97 MB)
   ✓ cmpd300_triplet.pt  (97 MB)

Resultados en Drive:
   ✓  14_loss_comparison.json

Logs de entrenamiento en Drive: ['train_arcface.log', 'train_ce.log', 'train_supcon.log', 'train_triplet.log']

✅ TODO GUARDADO — podés cerrar Colab tranquilo.
